In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "ASIS"
STUDENT_ID = "TUPM-26-1217"
SEED_DIGIT = int(STUDENT_ID[-1])
RANDOM_SEED = int("".join(STUDENT_ID.split("-")[1:]))

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

X_raw = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels_raw = np.genfromtxt(DATA_DIR / "secom_labels.data", dtype=str)
y = labels_raw[:, 0].astype(float).astype(int)

df = pd.DataFrame(X_raw, columns=[f"Feature_{i}" for i in range(X_raw.shape[1])])
df["Outcome"] = np.where(y == 1, "Fail", "Pass")

pass_mask = df["Outcome"] == "Pass"
fail_mask = df["Outcome"] == "Fail"

pass_counts = df.loc[pass_mask, df.columns[:-1]].notna().sum()
fail_counts = df.loc[fail_mask, df.columns[:-1]].notna().sum()
not_constant = df[df.columns[:-1]].std(ddof=1) > 0

usable_features = df.columns[:-1][
    (pass_counts >= 10) & (fail_counts >= 10) & not_constant
].to_numpy()

rng = np.random.default_rng(seed=RANDOM_SEED)
selected_features = np.sort(rng.choice(usable_features, size=5, replace=False)).tolist()

avail_pass_means = df.loc[pass_mask, selected_features].mean(skipna=True)
avail_fail_means = df.loc[fail_mask, selected_features].mean(skipna=True)
avail_diff = avail_fail_means - avail_pass_means

df_imputed = df[selected_features].fillna(df[selected_features].median())
imp_pass_means = df_imputed.loc[pass_mask].mean()
imp_fail_means = df_imputed.loc[fail_mask].mean()
imp_diff = imp_fail_means - imp_pass_means

n_boot = 200 + (10 * SEED_DIGIT)

def run_stratified_bootstrap(data_subset, pass_idx, fail_idx, n_reps, seed):
    boot_rng = np.random.default_rng(seed=seed)
    pass_arr = data_subset.loc[pass_idx, selected_features].to_numpy()
    fail_arr = data_subset.loc[fail_idx, selected_features].to_numpy()

    n_pass = pass_arr.shape[0]
    n_fail = fail_arr.shape[0]
    boot_diffs = np.empty((n_reps, len(selected_features)), dtype=float)

    for b in range(n_reps):
        p_sample = pass_arr[boot_rng.integers(0, n_pass, size=n_pass)]
        f_sample = fail_arr[boot_rng.integers(0, n_fail, size=n_fail)]

        boot_diffs[b] = np.nanmean(f_sample, axis=0) - np.nanmean(p_sample, axis=0)

    boot_mean = np.nanmean(boot_diffs, axis=0)
    ci_low = np.nanpercentile(boot_diffs, 2.5, axis=0)
    ci_high = np.nanpercentile(boot_diffs, 97.5, axis=0)
    return boot_mean, ci_low, ci_high

avail_boot_mean, avail_ci_low, avail_ci_high = run_stratified_bootstrap(
    df, pass_mask, fail_mask, n_boot, RANDOM_SEED
)
imp_boot_mean, imp_ci_low, imp_ci_high = run_stratified_bootstrap(
    df_imputed, pass_mask, fail_mask, n_boot, RANDOM_SEED
)

avail_boot_check, _, _ = run_stratified_bootstrap(df, pass_mask, fail_mask, n_boot, RANDOM_SEED)
imp_boot_check, _, _ = run_stratified_bootstrap(df_imputed, pass_mask, fail_mask, n_boot, RANDOM_SEED)
reproducibility_verified = bool(
    np.allclose(avail_boot_mean, avail_boot_check, equal_nan=True) and
    np.allclose(imp_boot_mean, imp_boot_check, equal_nan=True)
)

comparison_df = pd.DataFrame({
    "Feature": selected_features,
    "Avail_Diff": avail_diff.values,
    "Avail_Boot_Mean": avail_boot_mean,
    "Avail_CI_Low": avail_ci_low,
    "Avail_CI_High": avail_ci_high,
    "Imputed_Diff": imp_diff.values,
    "Imputed_Boot_Mean": imp_boot_mean,
    "Imputed_CI_Low": imp_ci_low,
    "Imputed_CI_High": imp_ci_high,
})

comparison_df["Abs_Diff_Change"] = (comparison_df["Imputed_Diff"] - comparison_df["Avail_Diff"]).abs()
comparison_df["Sign_Changed"] = np.sign(comparison_df["Avail_Diff"]) != np.sign(comparison_df["Imputed_Diff"])
comparison_df["Avail_Rank"] = comparison_df["Avail_Diff"].abs().rank(ascending=False).astype(int)
comparison_df["Imputed_Rank"] = comparison_df["Imputed_Diff"].abs().rank(ascending=False).astype(int)
comparison_df["Rank_Changed"] = comparison_df["Avail_Rank"] != comparison_df["Imputed_Rank"]

csv_out = OUTPUT_DIR / "robustness_comparison.csv"
txt_out = OUTPUT_DIR / "sensitivity_report.txt"
comparison_df.to_csv(csv_out, index=False)

largest_change_row = comparison_df.loc[comparison_df["Abs_Diff_Change"].idxmax()]
sign_changed_list = comparison_df.loc[comparison_df["Sign_Changed"], "Feature"].tolist()
rank_changed_list = comparison_df.loc[comparison_df["Rank_Changed"], "Feature"].tolist()

report_text = f"""======================================================================
      STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS REPORT
======================================================================
Student ID               : {STUDENT_ID}
Random Seed              : {RANDOM_SEED}
Selected Features        : {selected_features}
Bootstrap Repetitions    : {n_boot}
Largest Diff Change      : {largest_change_row['Feature']} ({largest_change_row['Abs_Diff_Change']:.6f})
Changed Difference Sign  : {sign_changed_list if sign_changed_list else 'None'}
Changed Feature Ranking  : {rank_changed_list if rank_changed_list else 'None'}
Reproducibility Verified : {reproducibility_verified}
======================================================================
"""

txt_out.write_text(report_text, encoding="utf-8")

print(report_text)
print(comparison_df.to_string(index=False))

      STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS REPORT
Student ID               : TUPM-26-1217
Random Seed              : 261217
Selected Features        : ['Feature_321', 'Feature_337', 'Feature_492', 'Feature_55', 'Feature_552']
Bootstrap Repetitions    : 270
Largest Diff Change      : Feature_492 (0.321726)
Changed Difference Sign  : None
Changed Feature Ranking  : ['Feature_321', 'Feature_337', 'Feature_492']
Reproducibility Verified : True

    Feature  Avail_Diff  Avail_Boot_Mean  Avail_CI_Low  Avail_CI_High  Imputed_Diff  Imputed_Boot_Mean  Imputed_CI_Low  Imputed_CI_High  Abs_Diff_Change  Sign_Changed  Avail_Rank  Imputed_Rank  Rank_Changed
Feature_321   -0.083085        -0.077450     -0.267823       0.086555     -0.082954          -0.077312       -0.267695         0.086644         0.000131         False           4             3          True
Feature_337    0.364312         0.380893     -0.129234       1.287736      0.365502           0.382045       -0.127495         1.2